[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/51_reward_shaping_solution.ipynb)

#  Solution: Reward Shaping & Process Rewards

Reference: Ng et al. "Policy invariance under reward transformations" (1999), OpenAI Process Reward Models (PRM), DeepSeek-R1 outcome-based rewards.

```text
1. Start with step_rewards (env signal, often sparse)
2. Add process_reward_fn(step_idx, context) * shaping_weight to each step
3. If done: add outcome_reward_fn(result) to final step
4. Return shaped tensor
```

In [ ]:
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch

In [ ]:
#  SOLUTION

def compute_shaped_reward(step_rewards, result, done, outcome_reward_fn,
                         process_reward_fn=None,
                         shaping_weight=1.0):
    """Compute shaped per-step rewards.

    Inspired by:
    - Ng et al. 1999: potential-based reward shaping preserves optimal policy
    - OpenAI PRM: process reward model gives credit per reasoning step
    - DeepSeek-R1: outcome-only reward (answer correctness + format)

    In agent RL, this addresses credit assignment: was step N good because
    it led to success, or was it lucky? Process rewards add intermediate signal.
    """
    shaped = torch.tensor(step_rewards, dtype=torch.float32).clone()

    # Add process rewards to intermediate steps
    if process_reward_fn is not None:
        for i in range(len(shaped)):
            shaped[i] += shaping_weight * process_reward_fn(i, None)

    # Add outcome reward to final step
    if done and callable(outcome_reward_fn):
        shaped[-1] += outcome_reward_fn(result)

    return shaped

In [ ]:
#  Verify
r = [0.0, 0.0, 0.0]
result = {"answer": "42", "correct": True}
shaped = compute_shaped_reward(r, result, True,
    lambda res: 5.0 if res["correct"] else -1.0,
    lambda i, ctx: 0.3 if i < 2 else 0.0)
print(f"Shaped: {shaped}")

In [ ]:
from torch_judge import check
check("reward_shaping")